In [19]:
import pandas as pd
import numpy as np

In [20]:
# ============================================
# MULTI-TOUCH MARKETING ATTRIBUTION PROJECT
# Owner: Beckley
# Solo Project - All 4 Weeks
# ============================================

# DATASET: final_shop_6modata
# Columns: Ad Group, Month, Impressions, Clicks,
# CTR, Conversions, Conv Rate, Cost, CPC,
# Revenue, Sale Amount, P&L

# 4 WEEK PLAN:
# Week 1: Load dataset, EDA, data cleaning
# Week 2: Attribution logic - First Touch, Last Touch, Linear
# Week 3: Calculate KPIs - ROAS, CAC, CPC
# Week 4: Dashboard visualization

In [21]:
# ============================================
# WEEK 1 - LOAD DATASET
# ============================================

df = pd.read_csv('data/final_shop_6modata.csv')

print(df.shape)
print(df.head())

(190, 12)
                                   Ad Group Month  Impressions  Clicks   CTR  \
0    Shop - 1:1 - Desk - [shop coupon code]  July        16038    6504  0.41   
1         Shop - 1:1 - Desk - [shop coupon]  July        36462   14367  0.39   
2  Shop - 1:1 - Desk - [shop discount code]  July         3635    1458  0.40   
3     Shop - 1:1 - Desk - [shop promo code]  July        26185   10418  0.40   
4          Shop - 1:1 - Desk - [shop promo]  July          808     282  0.35   

   Conversions  Conv Rate   Cost   CPC  Revenue  Sale Amount      P&L  
0         1166       0.10   6669  1.03     6402    136770.05 -267.086  
1         2188       0.09  13746  0.96    13262    283215.21 -483.951  
2          248       0.09   1606  1.10     1723     39165.46  117.136  
3         2294       0.12  13278  1.27    13042    284823.48 -235.921  
4           61       0.15    391  1.39      337      7717.77  -53.604  


In [22]:
# ============================================
# WEEK 1 - EXPLORE DATA TYPES AND NULLS
# ============================================

print(df.info())
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 190 entries, 0 to 189
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Ad Group     190 non-null    object 
 1   Month        190 non-null    object 
 2   Impressions  190 non-null    int64  
 3   Clicks       190 non-null    int64  
 4   CTR          190 non-null    float64
 5   Conversions  190 non-null    int64  
 6   Conv Rate    190 non-null    float64
 7   Cost         190 non-null    int64  
 8   CPC          190 non-null    float64
 9   Revenue      190 non-null    int64  
 10  Sale Amount  190 non-null    float64
 11  P&L          190 non-null    float64
dtypes: float64(5), int64(5), object(2)
memory usage: 17.9+ KB
None
Ad Group       0
Month          0
Impressions    0
Clicks         0
CTR            0
Conversions    0
Conv Rate      0
Cost           0
CPC            0
Revenue        0
Sale Amount    0
P&L            0
dtype: int64


In [23]:
# ============================================
# WEEK 1 - EXPLORE UNIQUE VALUES
# ============================================

# Unique months in dataset
print("Months in dataset:")
print(df['Month'].unique())

# Unique ad groups
print(f"\nTotal unique ad groups: {df['Ad Group'].nunique()}")
print("\nAd Groups:")
print(df['Ad Group'].unique())

Months in dataset:
['July' 'August' 'September' 'October' 'November']

Total unique ad groups: 40

Ad Groups:
['Shop - 1:1 - Desk - [shop coupon code]'
 'Shop - 1:1 - Desk - [shop coupon]'
 'Shop - 1:1 - Desk - [shop discount code]'
 'Shop - 1:1 - Desk - [shop promo code]'
 'Shop - 1:1 - Desk - [shop promo]'
 'Shop - 1:1 - Mob - [shop coupon code]'
 'Shop - 1:1 - Mob - [shop coupon]'
 'Shop - 1:1 - Mob - [shop discount code]'
 'Shop - 1:1 - Mob - [shop promo code]' 'Shop - 1:1 - Mob - [shop promo]'
 'Shop - Exact - Desk - Competitor' 'Shop - Exact - Desk - Coupon Code'
 'Shop - Exact - Desk - Discount Code'
 'Shop - Exact - Desk - Free Shipping' 'Shop - Exact - Desk - Offer'
 'Shop - Exact - Desk - Promo Code' 'Shop - Exact - Desk - Sale'
 'Shop - Exact - Mob - Competitor' 'Shop - Exact - Mob - Coupon Code'
 'Shop - Exact - Mob - Discount Code' 'Shop - Exact - Mob - Free Shipping'
 'Shop - Exact - Mob - Offer' 'Shop - Exact - Mob - Promo Code'
 'Shop - Exact - Mob - Sale' 'Shop - Phras

In [24]:
# ============================================
# WEEK 1 - BASIC STATISTICS
# ============================================

print(df.describe())

         Impressions        Clicks         CTR  Conversions   Conv Rate  \
count     190.000000    190.000000  190.000000   190.000000  190.000000   
mean    14077.363158   4865.805263    0.272105   505.242105    0.079737   
std     29771.686227  11348.529219    0.107894  1052.202922    0.052859   
min        35.000000      2.000000    0.050000     0.000000    0.000000   
25%      1065.000000    264.500000    0.192500    24.000000    0.050000   
50%      4969.000000    930.000000    0.285000    70.500000    0.070000   
75%     13380.000000   4190.500000    0.360000   428.250000    0.100000   
max    276568.000000  99526.000000    0.470000  7563.000000    0.500000   

               Cost         CPC       Revenue    Sale Amount          P&L  
count    190.000000  190.000000    190.000000     190.000000   190.000000  
mean    3344.063158    0.791263   2957.684211   63416.180579  -386.361916  
std     6524.606753    0.403312   5962.413097  125414.656922   903.073776  
min        1.000000 

In [25]:
# ============================================
# WEEK 2 - ATTRIBUTION LOGIC
# ============================================

# First, calculate ROAS per ad group per month
# ROAS = Revenue / Cost

df['ROAS'] = df['Revenue'] / df['Cost']

print(df[['Ad Group', 'Month', 'Cost', 'Revenue', 'ROAS']].head(10))

                                   Ad Group Month   Cost  Revenue      ROAS
0    Shop - 1:1 - Desk - [shop coupon code]  July   6669     6402  0.959964
1         Shop - 1:1 - Desk - [shop coupon]  July  13746    13262  0.964790
2  Shop - 1:1 - Desk - [shop discount code]  July   1606     1723  1.072852
3     Shop - 1:1 - Desk - [shop promo code]  July  13278    13042  0.982226
4          Shop - 1:1 - Desk - [shop promo]  July    391      337  0.861893
5     Shop - 1:1 - Mob - [shop coupon code]  July  13157     8550  0.649844
6          Shop - 1:1 - Mob - [shop coupon]  July  19371    13699  0.707191
7   Shop - 1:1 - Mob - [shop discount code]  July   2637     2038  0.772848
8      Shop - 1:1 - Mob - [shop promo code]  July  16946    14565  0.859495
9           Shop - 1:1 - Mob - [shop promo]  July    485      409  0.843299


In [26]:
# ============================================
# FIRST-TOUCH ATTRIBUTION
# ============================================

# First-Touch = credit goes to the FIRST month
# a customer saw an ad (earliest month = July)

# Get the first touch for each ad group
# Sort by month order first
month_order = ['July', 'August', 'September', 'October', 'November']
df['Month'] = pd.Categorical(df['Month'], categories=month_order, ordered=True)

# First touch = earliest month per ad group
first_touch = df.sort_values('Month').groupby('Ad Group').first().reset_index()

# Assign 100% of conversion credit to first touch
first_touch['Attribution_Model'] = 'First-Touch'
first_touch['Attribution_Weight'] = 1.0
first_touch['Attributed_Revenue'] = first_touch['Revenue']
first_touch['Attributed_Cost'] = first_touch['Cost']
first_touch['Attributed_ROAS'] = first_touch['Attributed_Revenue'] / first_touch['Attributed_Cost']

print(first_touch[['Ad Group', 'Month', 'Attributed_Revenue', 'Attributed_Cost', 'Attributed_ROAS']].head(10))

                                   Ad Group Month  Attributed_Revenue  \
0    Shop - 1:1 - Desk - [shop coupon code]  July                6402   
1         Shop - 1:1 - Desk - [shop coupon]  July               13262   
2  Shop - 1:1 - Desk - [shop discount code]  July                1723   
3     Shop - 1:1 - Desk - [shop promo code]  July               13042   
4          Shop - 1:1 - Desk - [shop promo]  July                 337   
5     Shop - 1:1 - Mob - [shop coupon code]  July                8550   
6          Shop - 1:1 - Mob - [shop coupon]  July               13699   
7   Shop - 1:1 - Mob - [shop discount code]  July                2038   
8      Shop - 1:1 - Mob - [shop promo code]  July               14565   
9           Shop - 1:1 - Mob - [shop promo]  July                 409   

   Attributed_Cost  Attributed_ROAS  
0             6669         0.959964  
1            13746         0.964790  
2             1606         1.072852  
3            13278         0.982226  
4     

In [27]:
# ============================================
# LAST-TOUCH ATTRIBUTION
# ============================================

# Last-Touch = credit goes to the LAST month
# a customer saw an ad (latest month = November)

# Last touch = latest month per ad group
last_touch = df.sort_values('Month').groupby('Ad Group').last().reset_index()

# Assign 100% of conversion credit to last touch
last_touch['Attribution_Model'] = 'Last-Touch'
last_touch['Attribution_Weight'] = 1.0
last_touch['Attributed_Revenue'] = last_touch['Revenue']
last_touch['Attributed_Cost'] = last_touch['Cost']
last_touch['Attributed_ROAS'] = last_touch['Attributed_Revenue'] / last_touch['Attributed_Cost']

print(last_touch[['Ad Group', 'Month', 'Attributed_Revenue', 
                   'Attributed_Cost', 'Attributed_ROAS']].head(10))

                                   Ad Group     Month  Attributed_Revenue  \
0    Shop - 1:1 - Desk - [shop coupon code]  November               16555   
1         Shop - 1:1 - Desk - [shop coupon]  November               23857   
2  Shop - 1:1 - Desk - [shop discount code]  November                3227   
3     Shop - 1:1 - Desk - [shop promo code]  November               34518   
4          Shop - 1:1 - Desk - [shop promo]  November                 910   
5     Shop - 1:1 - Mob - [shop coupon code]  November               24071   
6          Shop - 1:1 - Mob - [shop coupon]  November               32668   
7   Shop - 1:1 - Mob - [shop discount code]  November                4773   
8      Shop - 1:1 - Mob - [shop promo code]  November               42440   
9           Shop - 1:1 - Mob - [shop promo]  November                1121   

   Attributed_Cost  Attributed_ROAS  
0            18641         0.888096  
1            27336         0.872732  
2             3182         1.014142  


In [28]:
# ============================================
# LINEAR ATTRIBUTION
# ============================================

# Linear = credit split equally across ALL months
# 5 months in dataset so each month gets 20% (1/5)

# Count how many months each ad group appears in
month_counts = df.groupby('Ad Group')['Month'].count().reset_index()
month_counts.columns = ['Ad Group', 'Month_Count']

# Merge month counts back into main dataframe
linear = df.merge(month_counts, on='Ad Group')

# Each month gets equal share of credit
linear['Attribution_Model'] = 'Linear'
linear['Attribution_Weight'] = 1 / linear['Month_Count']
linear['Attributed_Revenue'] = linear['Revenue'] * linear['Attribution_Weight']
linear['Attributed_Cost'] = linear['Cost'] * linear['Attribution_Weight']

# Sum up attributed values per ad group
linear_summary = linear.groupby('Ad Group').agg(
    Attributed_Revenue=('Attributed_Revenue', 'sum'),
    Attributed_Cost=('Attributed_Cost', 'sum')
).reset_index()

linear_summary['Attribution_Model'] = 'Linear'
linear_summary['Attributed_ROAS'] = linear_summary['Attributed_Revenue'] / linear_summary['Attributed_Cost']

print(linear_summary[['Ad Group', 'Attributed_Revenue', 
                        'Attributed_Cost', 'Attributed_ROAS']].head(10))

                                   Ad Group  Attributed_Revenue  \
0    Shop - 1:1 - Desk - [shop coupon code]              7429.4   
1         Shop - 1:1 - Desk - [shop coupon]             12063.4   
2  Shop - 1:1 - Desk - [shop discount code]              1681.0   
3     Shop - 1:1 - Desk - [shop promo code]             14861.0   
4          Shop - 1:1 - Desk - [shop promo]               458.0   
5     Shop - 1:1 - Mob - [shop coupon code]             10589.2   
6          Shop - 1:1 - Mob - [shop coupon]             15295.0   
7   Shop - 1:1 - Mob - [shop discount code]              2288.8   
8      Shop - 1:1 - Mob - [shop promo code]             17634.2   
9           Shop - 1:1 - Mob - [shop promo]               480.8   

   Attributed_Cost  Attributed_ROAS  
0           8238.8         0.901758  
1          13839.6         0.871658  
2           1721.0         0.976758  
3          16297.0         0.911886  
4            494.8         0.925627  
5          11854.4         0.89327

In [29]:
# ============================================
# COMBINE ALL THREE ATTRIBUTION MODELS
# ============================================

# Prepare first touch summary
first_summary = first_touch[['Ad Group', 'Attribution_Model', 
                              'Attributed_Revenue', 'Attributed_Cost', 
                              'Attributed_ROAS']]

# Prepare last touch summary
last_summary = last_touch[['Ad Group', 'Attribution_Model', 
                            'Attributed_Revenue', 'Attributed_Cost', 
                            'Attributed_ROAS']]

# Prepare linear summary
linear_final = linear_summary[['Ad Group', 'Attribution_Model', 
                                'Attributed_Revenue', 'Attributed_Cost', 
                                'Attributed_ROAS']]

# Combine all three into one table
attribution_comparison = pd.concat([first_summary, last_summary, linear_final], 
                                    ignore_index=True)

# Sort by Ad Group for easy comparison
attribution_comparison = attribution_comparison.sort_values(['Ad Group', 
                                                             'Attribution_Model'])

print(attribution_comparison.head(15))
print(f"\nTotal rows: {attribution_comparison.shape[0]}")

                                    Ad Group Attribution_Model  \
0     Shop - 1:1 - Desk - [shop coupon code]       First-Touch   
40    Shop - 1:1 - Desk - [shop coupon code]        Last-Touch   
80    Shop - 1:1 - Desk - [shop coupon code]            Linear   
1          Shop - 1:1 - Desk - [shop coupon]       First-Touch   
41         Shop - 1:1 - Desk - [shop coupon]        Last-Touch   
81         Shop - 1:1 - Desk - [shop coupon]            Linear   
2   Shop - 1:1 - Desk - [shop discount code]       First-Touch   
42  Shop - 1:1 - Desk - [shop discount code]        Last-Touch   
82  Shop - 1:1 - Desk - [shop discount code]            Linear   
3      Shop - 1:1 - Desk - [shop promo code]       First-Touch   
43     Shop - 1:1 - Desk - [shop promo code]        Last-Touch   
83     Shop - 1:1 - Desk - [shop promo code]            Linear   
4           Shop - 1:1 - Desk - [shop promo]       First-Touch   
44          Shop - 1:1 - Desk - [shop promo]        Last-Touch   
84        